In [ ]:
!pip install -U pip
!pip uninstall -y matplotlib
!pip install -U matplotlib==3.8.4 numpy==1.24.4 ultralytics
!pip install tqdm

In [ ]:
# YOLOv12 Training Script
#
# This script trains a YOLOv12 model using a configuration defined in `config.yaml`.
# ## 1. Setup
# Import necessary libraries and define the path to the configuration file.
# =============================================================================
import yaml
import os
import pandas as pd
from itertools import product
from tqdm import tqdm
import math
import numpy as np
import subprocess
import importlib.util
import sys, platform
import torch
from pathlib import Path
import json 
import ultralytics
import matplotlib
matplotlib.use("Agg", force=True)
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import matplotlib.cm as cm
from typing import List, Dict, Tuple, Optional, Union
from PIL import Image
from matplotlib import font_manager
from ultralytics import YOLO
from ultralytics import settings 
from ultralytics.data.utils import check_det_dataset
import matplotlib as mpl, matplotlib.colors as colors
print(mpl.__version__, mpl.__file__)
print(colors.__file__)
assert hasattr(mpl, "colors")
#from ruamel.yaml import YAML


In [ ]:
# 1. Show NVIDIA driver / GPU info
# ────────────────────────────────────────────────────
import subprocess, sys, textwrap, importlib, os
import torch

print("=== nvidia-smi ===")
try:
    smi_output = subprocess.check_output(["nvidia-smi"], text=True)
    print(smi_output)
except FileNotFoundError:
    print("nvidia-smi not found ➜ NVIDIA driver likely missing or PATH issue.")
except subprocess.CalledProcessError as err:
    print("nvidia-smi returned an error:\n", err.output)

# ────────────────────────────────────────────────────
# 2. PyTorch & CUDA status
# ────────────────────────────────────────────────────
print("\n=== PyTorch / CUDA ===")
print(f"PyTorch: {torch.__version__}")
print(f"Built with CUDA: {torch.version.cuda}")
print(f"CUDA available at runtime: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU device: {torch.cuda.get_device_name(0)}")

In [ ]:
# 2.  Helper functions
# ──────────────────────────────────────────────────────────────────────────────
def configure_ultralytics_datasets_root(root: Path) -> None:
    """Stick Ultralytics datasets_dir to <project_root>."""
    settings.update({"datasets_dir": str(root)})
    print(f"✅  datasets_dir → {settings['datasets_dir']}")

def tidy_dataset_yaml(yaml_path: Path, single_class_name: str | None = None) -> None:
    """Remove 'path:' entry & enforce nc/names consistency."""
    data = yaml.safe_load(yaml_path.read_text())
    if data.pop("path", None) is not None:
        print("🔁  Removed stale 'path:' key from dataset YAML")
    if single_class_name:
        data["nc"] = 1
        data["names"] = [single_class_name]
    yaml_path.write_text(yaml.safe_dump(data, sort_keys=False))
    print(f"✅  Dataset YAML saved: {yaml_path.relative_to(Path().resolve())}")

def load_config(cfg_path: str) -> dict:
    cfg = yaml.safe_load(Path(cfg_path).read_text())
    if not cfg:
        raise RuntimeError("Empty configuration file")
    return cfg

In [ ]:
# 3.  Banner & environment info
# ──────────────────────────────────────────────────────────────────────────────
print("🐍", sys.version.split()[0], "| torch", torch.__version__, "|", platform.platform())
print("numpy", np.__version__)

print("Ultralytics:", ultralytics.__version__)
print("CUDA available:", torch.cuda.is_available(), "\n")

project_root = Path().resolve()
configure_ultralytics_datasets_root(project_root)

In [ ]:
# 5.  Prepare dataset YAML and verify folder layout
CONFIG_PATH = project_root / 'config.yaml'
cfg = load_config(CONFIG_PATH)
#change the path name as needed
dataset_yaml_path = Path(cfg["sleeve_yaml_path"]).resolve()
tidy_dataset_yaml(dataset_yaml_path, single_class_name="sleeve")
check_det_dataset(str(dataset_yaml_path))
print("✅  Dataset structure verified by Ultralytics\n")

print("🚀  Initialising model:", model_type)
model = YOLO(model_type)
print("✅  Model ready\n")

In [ ]:
# Ensemble up to 4 YOLO (Ultralytics) models with per-model conf/IoU/augment/imgsz,
# fuse detections (WBF or NMS), and evaluate against YOLO-format ground truth.
# Designed for single-class detection but works multi-class too.

# -----------------------------
# Utility: IOU, NMS, WBF
# -----------------------------
def box_iou_xyxy(a: np.ndarray, b: np.ndarray) -> np.ndarray:
    """
    IoU between two sets of boxes in xyxy format.
    a: (N,4), b: (M,4)
    returns: (N,M) IoU matrix
    """
    if a.size == 0 or b.size == 0:
        return np.zeros((a.shape[0], b.shape[0]), dtype=np.float32)
    # areas
    a_wh = np.clip(a[:, 2:4] - a[:, 0:2], a_min=0, a_max=None)
    b_wh = np.clip(b[:, 2:4] - b[:, 0:2], a_min=0, a_max=None)
    a_area = a_wh[:, 0] * a_wh[:, 1]
    b_area = b_wh[:, 0] * b_wh[:, 1]

    # intersections
    lt = np.maximum(a[:, None, 0:2], b[None, :, 0:2])
    rb = np.minimum(a[:, None, 2:4], b[None, :, 2:4])
    wh = np.clip(rb - lt, a_min=0, a_max=None)
    inter = wh[:, :, 0] * wh[:, :, 1]
    union = a_area[:, None] + b_area[None, :] - inter
    iou = np.where(union > 0, inter / (union + 1e-9), 0.0)
    return iou.astype(np.float32)

def nms_xyxy(boxes: np.ndarray, scores: np.ndarray, iou_thr: float) -> List[int]:
    """
    Simple NMS for xyxy boxes. Returns indices to keep.
    """
    if boxes.size == 0:
        return []
    order = scores.argsort()[::-1]
    keep = []
    while order.size > 0:
        i = order[0]
        keep.append(i)
        if order.size == 1:
            break
        ious = box_iou_xyxy(boxes[i:i+1], boxes[order[1:]])[0]
        remain = np.where(ious <= iou_thr)[0]
        order = order[1:][remain]
    return keep

def wbf_xyxy(
    boxes_list: List[np.ndarray],
    scores_list: List[np.ndarray],
    labels_list: List[np.ndarray],
    weights: Optional[List[float]] = None,
    iou_thr: float = 0.55,
    skip_box_thr: float = 0.0
) -> Tuple[np.ndarray, np.ndarray, np.ndarray]:
    """
    Very light-weight Weighted Boxes Fusion (single or multi-class).
    - boxes_list, scores_list, labels_list: lists over models; each element is (N_i,4)/(N_i,)/(N_i,)
    - weights: per-model weights for fusion; defaults to 1.0 each
    Returns fused (boxes, scores, labels)
    Notes:
      * Greedy clustering by IoU on the fly; order by score desc across all models.
      * For each cluster (same class), coords = weighted average by (score * model_weight).
      * Score = average of member scores (weighted).
    """
    # Flatten all detections into one list with model_id
    all_entries = []
    for m, (b, s, l) in enumerate(zip(boxes_list, scores_list, labels_list)):
        if b.size == 0:
            continue
        for i in range(b.shape[0]):
            if s[i] >= skip_box_thr:
                all_entries.append((m, float(s[i]), int(l[i]), b[i].astype(np.float32)))

    if not all_entries:
        return (np.zeros((0,4), np.float32), np.zeros((0,), np.float32), np.zeros((0,), np.int32))

    # Sort by score desc to seed clusters with strong boxes
    all_entries.sort(key=lambda x: x[1], reverse=True)

    if weights is None:
        weights = [1.0] * len(boxes_list)

    clusters = []  # each cluster: dict with 'class', 'members': list[(m,score,box)], and running 'box'/'score' accumulators
    for m_id, score, cls, box in all_entries:
        placed = False
        for c in clusters:
            if c['class'] != cls:
                continue
            # IoU with current cluster average
            iou = box_iou_xyxy(box[None, :], c['box'][None, :])[0, 0]
            if iou >= iou_thr:
                w = score * weights[m_id]
                # update weighted average coords
                c['sum_w'] += w
                c['sum_ws'] += w * score
                c['sum_wx1'] += w * box[0]
                c['sum_wy1'] += w * box[1]
                c['sum_wx2'] += w * box[2]
                c['sum_wy2'] += w * box[3]
                # refresh average box
                c['box'][0] = c['sum_wx1'] / (c['sum_w'] + 1e-9)
                c['box'][1] = c['sum_wy1'] / (c['sum_w'] + 1e-9)
                c['box'][2] = c['sum_wx2'] / (c['sum_w'] + 1e-9)
                c['box'][3] = c['sum_wy2'] / (c['sum_w'] + 1e-9)
                c['members'].append((m_id, score, box))
                placed = True
                break
        if not placed:
            w = score * weights[m_id]
            clusters.append({
                'class': cls,
                'members': [(m_id, score, box)],
                'sum_w': w,
                'sum_ws': w * score,
                'sum_wx1': w * box[0],
                'sum_wy1': w * box[1],
                'sum_wx2': w * box[2],
                'sum_wy2': w * box[3],
                'box': box.copy()
            })

    fused_boxes, fused_scores, fused_labels = [], [], []
    for c in clusters:
        # Weighted-average score (normalized by sum_w of weights*scores)
        # Using classic WBF scoring alternative: mean of member scores (weighted by weights*score)
        score = c['sum_ws'] / (c['sum_w'] + 1e-9)
        fused_boxes.append(c['box'])
        fused_scores.append(score)
        fused_labels.append(c['class'])

    return (
        np.vstack(fused_boxes).astype(np.float32),
        np.array(fused_scores, dtype=np.float32),
        np.array(fused_labels, dtype=np.int32),
    )

In [ ]:
# -----------------------------
# Load dataset + ground truth
# -----------------------------
IMG_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".tif", ".tiff", ".webp"}

def _list_images_from_path(p: str):
    p = str(p)
    if os.path.isfile(p) and p.lower().endswith(".txt"):
        with open(p, "r") as f:
            return [l.strip() for l in f if l.strip()]
    if os.path.isdir(p):
        out = []
        for root, _, files in os.walk(p):
            for fn in files:
                if Path(fn).suffix.lower() in IMG_EXTS:
                    out.append(str(Path(root) / fn))
        out.sort()
        return out
    # glob or single file
    candidates = sorted([str(x) for x in Path().glob(p)])
    return [c for c in candidates if Path(c).suffix.lower() in IMG_EXTS]

def _image_to_label_path(image_path: str) -> str:
    """
    Convert .../images/.../name.jpg -> .../labels/.../name.txt
    If 'images' not in path, try sibling 'labels' folder.
    """
    ip = Path(image_path)
    if "images" in ip.parts:
        parts = list(ip.parts)
        idx = parts.index("images")
        parts[idx] = "labels"
        lbl = Path(*parts).with_suffix(".txt")
        return str(lbl)
    # fallback: same dir, replace ext with .txt
    return str(ip.with_suffix(".txt"))

def _read_yolo_labels(label_path: str, img_w: int, img_h: int) -> Tuple[np.ndarray, np.ndarray]:
    """
    Read YOLO txt: cls cx cy w h (normalized). Return xyxy (N,4), labels (N,)
    """
    if not os.path.exists(label_path):
        return (np.zeros((0,4), dtype=np.float32), np.zeros((0,), dtype=np.int32))
    arr = []
    with open(label_path, "r") as f:
        for line in f:
            parts = line.strip().split()
            if len(parts) < 5:
                continue
            cls = int(float(parts[0]))
            cx, cy, w, h = map(float, parts[1:5])
            x1 = (cx - w/2.0) * img_w
            y1 = (cy - h/2.0) * img_h
            x2 = (cx + w/2.0) * img_w
            y2 = (cy + h/2.0) * img_h
            arr.append((cls, x1, y1, x2, y2))
    if not arr:
        return (np.zeros((0,4), dtype=np.float32), np.zeros((0,), dtype=np.int32))
    arr = np.array(arr, dtype=np.float32)
    labels = arr[:, 0].astype(np.int32)
    boxes = arr[:, 1:5]
    return (boxes, labels)

def load_split_images_from_yaml(data_yaml: str, split: str = "test"):
    with open(data_yaml, "r") as f:
        y = yaml.safe_load(f) or {}

    names = y.get("names", {})
    if isinstance(names, list):
        names = {i: n for i, n in enumerate(names)}

    yaml_dir = Path(data_yaml).parent
    root = Path(y.get("path", yaml_dir)).expanduser().resolve()

    synonyms = {
        "train": ["train", "training"],
        "val":   ["val", "valid", "validation"],
        "test":  ["test", "testing", "eval", "evaluation"],
    }
    req = split.lower().strip()
    raw = None
    for k in synonyms.get(req, [req]):
        if k in y and y[k]:
            raw = str(y[k]).strip()
            break
    if raw is None:
        raise ValueError(f"No '{split}' entry found in {data_yaml}.")

    tried = []
    def _try(p: Path):
        tried.append(str(p))
        return _list_images_from_path(str(p))

    p = Path(raw)
    if not p.is_absolute():
        p = (root / raw)

    imgs = _try(p)
    if not imgs and p.is_dir():
        imgs = _try(p / "images")

    # common patterns
    if not imgs:
        imgs = _try(root / "images" / req)
    if not imgs:
        imgs = _try(root / req / "images")

    if not imgs and yaml_dir != root:
        imgs = _try(yaml_dir / raw)
    if not imgs and (yaml_dir / raw).is_dir():
        imgs = _try(yaml_dir / raw / "images")
    if not imgs:
        imgs = _try(yaml_dir / "images" / req)
    if not imgs:
        imgs = _try(yaml_dir / req / "images")

    if not imgs:
        raise ValueError(
            f"No {split} images found.\n"
            f" data_yaml: {data_yaml}\n"
            f" path root: {root}\n"
            " Tried:\n  - " + "\n  - ".join(tried)
        )
    return imgs, names, req

In [ ]:
# -----------------------------
# Inference per model
# -----------------------------
def run_model_predictions(
    weights: str,
    image_paths: List[str],
    conf: float = 0.25,
    iou: float = 0.7,
    imgsz: int = 640,
    augment: bool = False,
    device: Optional[str] = None,
    half: bool = False,
    workers=-1
):
    """
    Run ultralytics model on a list of image paths.
    Returns list (len=image_paths) of dicts with keys: boxes(N,4), scores(N,), labels(N,)
    """
    model = YOLO(weights)
    preds_per_image = []

    # Batch predict for speed; Ultralytics keeps order of inputs in results
    results = model.predict(
        source=image_paths,
        conf=conf,
        iou=iou,
        imgsz=imgsz,
        augment=augment,
        device=device,
        half=half,
        verbose=False,
        plots=True,
        workers=-1
    )
    # results is a list of ultralytics.engine.results.Results (one per input image)
    for res in results:
        b = res.boxes
        if b is None or b.shape[0] == 0:
            preds_per_image.append({
                "boxes": np.zeros((0,4), dtype=np.float32),
                "scores": np.zeros((0,), dtype=np.float32),
                "labels": np.zeros((0,), dtype=np.int32),
                "shape": res.orig_shape  # (H,W)
            })
            continue
        boxes = b.xyxy.cpu().numpy().astype(np.float32)
        scores = b.conf.cpu().numpy().astype(np.float32)
        labels = b.cls.cpu().numpy().astype(np.int32)
        preds_per_image.append({
            "boxes": boxes,
            "scores": scores,
            "labels": labels,
            "shape": res.orig_shape  # (H,W)
        })
    return preds_per_image

# -----------------------------
# Ensemble per image
# -----------------------------
def ensemble_image_predictions(
    per_model_preds: List[Dict[str, np.ndarray]],
    method: str = "wbf",
    weights: Optional[List[float]] = None,
    ensemble_iou: float = 0.55,
    final_nms_iou: Optional[float] = None,
    score_thr: float = 0.0
) -> Dict[str, np.ndarray]:
    """
    Fuse predictions from multiple models for a single image.
    - method: 'wbf' (default) or 'nms' (union-then-NMS)
    - weights: per-model weights for WBF
    - ensemble_iou: IoU threshold for WBF grouping (or for union-NMS)
    - final_nms_iou: optional NMS after WBF to remove near-duplicates
    - score_thr: drop boxes below score after fusion
    """
    boxes_list, scores_list, labels_list = [], [], []
    for p in per_model_preds:
        boxes_list.append(p["boxes"])
        scores_list.append(p["scores"])
        labels_list.append(p["labels"])

    if method == "nms":
        # Union all boxes then NMS
        boxes = np.concatenate(boxes_list, axis=0) if boxes_list else np.zeros((0,4), np.float32)
        scores = np.concatenate(scores_list, axis=0) if scores_list else np.zeros((0,), np.float32)
        labels = np.concatenate(labels_list, axis=0) if labels_list else np.zeros((0,), np.int32)
    else:
        # Default WBF
        boxes, scores, labels = wbf_xyxy(
            boxes_list, scores_list, labels_list,
            weights=weights, iou_thr=ensemble_iou, skip_box_thr=0.0
        )

    # optional final NMS per class (useful after WBF too)
    if final_nms_iou is not None and boxes.shape[0] > 0:
        keep_all = []
        for cls in np.unique(labels):
            idx = np.where(labels == cls)[0]
            if idx.size == 0:
                continue
            k = nms_xyxy(boxes[idx], scores[idx], final_nms_iou)
            keep_all.extend(idx[k])
        keep_all = np.array(keep_all, dtype=int)
        boxes, scores, labels = boxes[keep_all], scores[keep_all], labels[keep_all]

    if score_thr is not None and boxes.shape[0] > 0:
        sel = scores >= float(score_thr)
        boxes, scores, labels = boxes[sel], scores[sel], labels[sel]

    return {"boxes": boxes, "scores": scores, "labels": labels}

In [ ]:
# -----------------------------
# Evaluation
# -----------------------------
def match_predictions_to_gt(
    preds: Dict[str, np.ndarray],
    gt_boxes: np.ndarray,
    gt_labels: np.ndarray,
    iou_match: float = 0.5,
) -> Tuple[int, int, int]:
    """
    Greedy one-to-one matching at IoU >= iou_match, per class.
    Returns TP, FP, FN counts for this image (box-level).
    """
    pred_boxes = preds["boxes"]
    pred_labels = preds["labels"]
    if pred_boxes.size == 0 and gt_boxes.size == 0:
        return 0, 0, 0
    if pred_boxes.size == 0:
        return 0, 0, gt_boxes.shape[0]
    if gt_boxes.size == 0:
        return 0, pred_boxes.shape[0], 0

    # class-wise matching
    TP, FP = 0, 0
    matched_gt = np.zeros((gt_boxes.shape[0],), dtype=bool)
    # sort preds by score desc for stable matching
    order = np.argsort(-preds["scores"])
    for i in order:
        pbox = pred_boxes[i:i+1]
        pcl  = pred_labels[i]
        idxs = np.where(gt_labels == pcl)[0]
        if idxs.size == 0:
            FP += 1
            continue
        ious = box_iou_xyxy(pbox, gt_boxes[idxs])[0]
        # choose best GT (unmatched) above threshold
        best = np.argmax(ious)
        best_iou = ious[best]
        gt_idx = idxs[best]
        if best_iou >= iou_match and not matched_gt[gt_idx]:
            matched_gt[gt_idx] = True
            TP += 1
        else:
            FP += 1
    FN = int((~matched_gt).sum())
    return TP, FP, FN

def compute_presence_confusion(preds_list: List[Dict[str, np.ndarray]], gts_list: List[Tuple[np.ndarray, np.ndarray]]) -> Dict[str, int]:
    """
    Presence-based confusion matrix at image level:
    TP_img: any GT present and any pred present
    TN_img: no GT and no pred
    FP_img: no GT but pred present
    FN_img: GT present but no pred
    """
    TP_img = TN_img = FP_img = FN_img = 0
    for preds, (gt_boxes, _) in zip(preds_list, gts_list):
        has_pred = preds["boxes"].shape[0] > 0
        has_gt = gt_boxes.shape[0] > 0
        if has_gt and has_pred:
            TP_img += 1
        elif (not has_gt) and (not has_pred):
            TN_img += 1
        elif (not has_gt) and has_pred:
            FP_img += 1
        else:  # has_gt and no pred
            FN_img += 1
    return dict(TP_img=TP_img, 
                TN_img=TN_img, 
                FP_img=FP_img, 
                FN_img=FN_img)

def pr_ap_curve(
    all_pred: List[Dict[str, np.ndarray]],
    all_gt: List[Tuple[np.ndarray, np.ndarray]],
    iou_match: float = 0.5
) -> Tuple[np.ndarray, np.ndarray, float]:
    """
    Compute Precision-Recall and AP (VOC 2010 101-point interp) at given IoU.
    Collate all predictions across dataset, then greedy match to GT.
    """
    # Build a global list of predictions with image index
    flat_preds = []
    total_gt = 0
    for img_idx, (preds, (gt_boxes, gt_labels)) in enumerate(zip(all_pred, all_gt)):
        total_gt += gt_boxes.shape[0]
        for i in range(preds["boxes"].shape[0]):
            flat_preds.append((img_idx, preds["scores"][i], preds["labels"][i], preds["boxes"][i]))
    if len(flat_preds) == 0:
        # no predicted boxes at all
        recalls = np.array([0.0, 1.0])
        precisions = np.array([1.0, 0.0])
        return precisions, recalls, 0.0

    # sort by score desc
    flat_preds.sort(key=lambda x: x[1], reverse=True)

    # prepare matched flags per image
    matched = [np.zeros((all_gt[i][0].shape[0],), dtype=bool) for i in range(len(all_gt))]

    tps, fps = [], []
    for (img_idx, score, cls, pbox) in flat_preds:
        gt_boxes, gt_labels = all_gt[img_idx]
        # class filter
        idxs = np.where(gt_labels == cls)[0]
        if idxs.size == 0:
            tps.append(0); fps.append(1)
            continue
        ious = box_iou_xyxy(pbox[None, :], gt_boxes[idxs])[0]
        best = int(np.argmax(ious))
        best_iou = float(ious[best])
        gt_idx = idxs[best]
        if best_iou >= iou_match and not matched[img_idx][gt_idx]:
            matched[img_idx][gt_idx] = True
            tps.append(1); fps.append(0)
        else:
            tps.append(0); fps.append(1)

    tps = np.cumsum(np.array(tps))
    fps = np.cumsum(np.array(fps))
    recalls = tps / (total_gt + 1e-9)
    precisions = tps / np.maximum(tps + fps, 1e-9)

    # 101-point interpolation
    recall_points = np.linspace(0, 1, 101)
    precis_at_r = []
    for r in recall_points:
        precis_at_r.append(np.max(precisions[recalls >= r]) if np.any(recalls >= r) else 0.0)
    ap = float(np.mean(precis_at_r))
    return precisions, recalls, ap

def evaluate_predictions(
    preds_list: List[Dict[str, np.ndarray]],
    images: List[str],
    iou_match_list: List[float],
) -> Dict[float, Dict[str, float]]:
    """
    Evaluate predictions against GT at multiple IoU thresholds (e.g., [0.5] or [0.5..0.95]).
    Returns: {iou_thr: {"TP":..., "FP":..., "FN":..., "precision":..., "recall":..., "f1":..., "AP":..., "mAP":...}}
    """
    # Load GT for all images (once)
    gts = []
    for img_path in images:
        with Image.open(img_path) as im:
            w, h = im.size
        label_path = _image_to_label_path(img_path)
        gt_boxes, gt_labels = _read_yolo_labels(label_path, w, h)
        gts.append((gt_boxes, gt_labels))

    # Presence confusion matrix (image-level)
    presence_cm = compute_presence_confusion(preds_list, gts)

    metrics = {}
    for thr in iou_match_list:
        # Box-level TP/FP/FN
        TP = FP = FN = 0
        for preds, (gtb, gtl) in zip(preds_list, gts):
            tpi, fpi, fni = match_predictions_to_gt(preds, gtb, gtl, iou_match=thr)
            TP += tpi; FP += fpi; FN += fni
        prec = TP / max(TP + FP, 1e-9)
        rec  = TP / max(TP + FN, 1e-9)
        f1   = 2 * prec * rec / max(prec + rec, 1e-9) if (prec + rec) > 0 else 0.0

        # AP at this IoU
        P, R, AP = pr_ap_curve(preds_list, gts, iou_match=thr)
        metrics[thr] = {
            "TP": int(TP), "FP": int(FP), "FN": int(FN),
            "precision": float(prec), "recall": float(rec), "f1": float(f1),
            "AP": float(AP),
            "presence_TP": presence_cm["TP_img"],
            "presence_TN": presence_cm["TN_img"],
            "presence_FP": presence_cm["FP_img"],
            "presence_FN": presence_cm["FN_img"],
        }
    # mAP if multiple thresholds
    if len(iou_match_list) > 1:
        metrics["mAP"] = float(np.mean([metrics[t]["AP"] for t in iou_match_list]))
    return metrics

In [ ]:
def _save_ensemble_yolo_txt(images, ensemble_preds, out_dir):
    out = Path(out_dir) / "preds_ensemble"
    out.mkdir(parents=True, exist_ok=True)
    for img_path, pred in zip(images, ensemble_preds):
        p = Path(img_path)
        stem = p.stem
        with Image.open(img_path) as im:
            w, h = im.size
        # xyxy -> cxcywh (normalized)
        b = pred["boxes"].astype(np.float32)
        if b.size == 0:
            (out / f"{stem}.txt").write_text("")  # empty file
            continue
        cx = (b[:,0] + b[:,2]) / 2.0 / w
        cy = (b[:,1] + b[:,3]) / 2.0 / h
        ww = (b[:,2] - b[:,0]) / w
        hh = (b[:,3] - b[:,1]) / h
        cls = pred["labels"].astype(int)
        lines = [f"{int(c)} {cx[i]:.6f} {cy[i]:.6f} {ww[i]:.6f} {hh[i]:.6f}" for i,c in enumerate(cls)]
        (out / f"{stem}.txt").write_text("\n".join(lines))

_save_ensemble_yolo_txt(results["images"], ensemble_preds, results["output_dir"])

In [ ]:
# -----------------------------
# Orchestrator
# -----------------------------
def run_ensemble_evaluation(
    data_yaml: str,
    models_cfg: List[Dict],
    method: str = "wbf",
    ensemble_iou: float = 0.55,
    final_nms_iou: Optional[float] = None,
    score_thr: float = 0.0,
    iou_match_list: Optional[List[float]] = None,
    device: Optional[str] = None,
    half: bool = False,
    data_split: str = "test",              # 'test' or 'val' (or 'train' if you want)
    output_dir: Optional[str] = None,      # directory to save artifacts
):

    if iou_match_list is None:
        iou_match_list = [0.5]

    # (A) load requested split
    images, class_names, used_key = load_split_images_from_yaml(data_yaml, split=data_split)
    print(f"Found {len(images)} images from split '{used_key}' in {data_yaml}.")

    # run each model with its own params (unchanged)
    all_model_preds = []
    for m in models_cfg:
        print(f"\nRunning model: {m['weights']}")
        preds = run_model_predictions(
            weights=m["weights"],
            image_paths=images,
            conf=float(m.get("conf", 0.25)),
            iou=float(m.get("iou", 0.7)),
            imgsz=int(m.get("imgsz", 640)),
            augment=bool(m.get("augment", False)),
            device=device,
            half=half,
        )
        all_model_preds.append(preds)

    # evaluate each single model (unchanged)
    per_model_metrics = []
    for i, preds in enumerate(all_model_preds):
        name = Path(models_cfg[i]["weights"]).stem
        metrics = evaluate_predictions(preds, images, iou_match_list)
        per_model_metrics.append((name, metrics))

    # fuse
    weights = [float(m.get("weight", 1.0)) for m in models_cfg]
    ensemble_preds = []
    print("\nFusing predictions...")
    for idx in tqdm(range(len(images))):
        per_img_per_model = [mp[idx] for mp in all_model_preds]
        fused = ensemble_image_predictions(
            per_img_per_model,
            method=method,
            weights=weights,
            ensemble_iou=ensemble_iou,
            final_nms_iou=final_nms_iou,
            score_thr=score_thr
        )
        ensemble_preds.append(fused)

    # evaluate ensemble (unchanged)
    ensemble_metrics = evaluate_predictions(ensemble_preds, images, iou_match_list)

    # summarize (unchanged)
    rows = []
    def _metrics_to_row(tag: str, m: Dict) -> List[Dict]:
        out = []
        keys = [k for k in m.keys() if isinstance(k, float)]
        keys = sorted(keys)
        row = {"Model": tag}
        for thr in keys:
            row.update({
                f"TP@{thr:.2f}": m[thr]["TP"],
                f"FP@{thr:.2f}": m[thr]["FP"],
                f"FN@{thr:.2f}": m[thr]["FN"],
                f"Precision@{thr:.2f}": m[thr]["precision"],
                f"Recall@{thr:.2f}": m[thr]["recall"],
                f"F1@{thr:.2f}": m[thr]["f1"],
                f"AP@{thr:.2f}": m[thr]["AP"],
                f"Pres_TP": m[thr]["presence_TP"],
                f"Pres_TN": m[thr]["presence_TN"],
                f"Pres_FP": m[thr]["presence_FP"],
                f"Pres_FN": m[thr]["presence_FN"],
            })
        if "mAP" in m:
            row["mAP"] = m["mAP"]
        out.append(row)
        return out

    for name, mm in per_model_metrics:
        rows.extend(_metrics_to_row(name, mm))
    rows.extend(_metrics_to_row("Ensemble", ensemble_metrics))
    df = pd.DataFrame(rows).fillna("")

    # (B) optional outputs to disk
    out_dir = None
    if output_dir is not None:
        out_dir = Path(output_dir)
        out_dir.mkdir(parents=True, exist_ok=True)
        # CSV summary
        df.to_csv(out_dir / "summary.csv", index=False)
        # JSON metrics
        # make them JSON-friendly
        def _to_py(o):
            import numpy as np
            if isinstance(o, (np.floating,)):
                return float(o)
            if isinstance(o, (np.integer,)):
                return int(o)
            return o
        with open(out_dir / "ensemble_metrics.json", "w") as f:
            json.dump(ensemble_metrics, f, default=_to_py, indent=2)
        with open(out_dir / "per_model_metrics.json", "w") as f:
            json.dump({name: m for name, m in per_model_metrics}, f, default=_to_py, indent=2)
        # (optional) record which split/path was used
        with open(out_dir / "run_meta.json", "w") as f:
            meta = {
                "data_yaml": str(data_yaml),          
                "split_requested": str(data_split),
                "split_used": str(used_key)
            }
            json.dump(meta, f, indent=2)

    return {
        "per_model_metrics": per_model_metrics,
        "ensemble_metrics": ensemble_metrics,
        "summary_df": df,
        "images": images,
        "split_used": used_key,
        "output_dir": str(out_dir) if out_dir else None
    }

In [ ]:
CONFIG_PATH = project_root / 'config.yaml'
cfg = load_config(CONFIG_PATH)
dataset_yaml_path = Path(cfg["sleeve_yaml_path"]).resolve()
#data_yaml = "data/v6/sleeves_v5_yolo/dataset.yaml"

models_cfg = [
    {
        "weights": "results/v6/YOLOv12l/sleeves_v6_best_2/weights/best.pt",
        "conf": 0.6, "iou": 0.6, "augment": True,
        "imgsz": 640, "weight": 1.075
    },
    {
        "weights": "results/v6/YOLOv12l/sleeves_v6_best_2/weights/best.pt",
        "conf": 0.48, "iou": 0.6, "augment": False,
        "imgsz": 640, "weight": 1.25
    },
    {
      "weights": "results/v6/YOLOv12l/sleeves_v6_best_1/weights/best.pt",
       "conf": 0.458, "iou": 0.2, "augment": False,
       "imgsz": 640, "weight": 0.75
     },
    {
       "weights": "results/v6/YOLOv12l/sleeves_v6_best_1/weights/best.pt",
       "conf": 0.49, "iou": 0.4, "augment": True,
       "imgsz": 640, "weight": 1.125
     }
]

results4 = run_ensemble_evaluation(
    data_yaml=dataset_yaml_path,
    models_cfg=models_cfg,
    method="wbf",           # or "nms"
    ensemble_iou=0.1,      # fusion IoU (WBF grouping or union-NMS)
    final_nms_iou=None,      # optional NMS after fusion (set None to skip)
    score_thr=0.55,          # drop fused boxes below this score
    iou_match_list=[0.1],  # evaluate at AP50 and AP75; add more (e.g., np.arange(0.5,0.96,0.05).tolist())
    device=0,               # or "cpu"
    half=False,
    data_split='test',
    output_dir="results/ensembles/test_4mods"
)

In [ ]:
models_cfg = [
    {
        "weights": "results/v6/YOLOv12l/sleeves_v6_best_2/weights/best.pt",
        "conf": 0.45, "iou": 0.6, "augment": True,
        "imgsz": 640, "weight": 1.075
    },
    {
        "weights": "results/v6/YOLOv12l/sleeves_v6_best_2/weights/best.pt",
        "conf": 0.45, "iou": 0.6, "augment": False,
        "imgsz": 640, "weight": 1.25
    },
    {
       "weights": "results/v6/YOLOv12l/sleeves_v6_best_1/weights/best.pt",
       "conf": 0.45, "iou": 0.2, "augment": False,
       "imgsz": 640, "weight": 0.75
     },
    {
       "weights": "results/v6/YOLOv12l/sleeves_v6_best_1/weights/best.pt",
       "conf": 0.45, "iou": 0.4, "augment": True,
       "imgsz": 640, "weight": 1.125
     }
]

results6 = run_ensemble_evaluation(
    data_yaml=dataset_yaml_path,
    models_cfg=models_cfg,
    method="wbf",           # or "nms"
    ensemble_iou=0.1,      # fusion IoU (WBF grouping or union-NMS)
    final_nms_iou=None,      # optional NMS after fusion (set None to skip)
    score_thr=0.55,          # drop fused boxes below this score
    iou_match_list=[0.1],  # evaluate at AP50 and AP75; add more (e.g., np.arange(0.5,0.96,0.05).tolist())
    device=0,               # or "cpu"
    half=False,
    data_split='test',
    output_dir="results/ensembles/val_v16_scr_0.55"
)

In [ ]:
print("0.54 - final conf, 0.45 indiv")
display(results5["summary_df"])
print("0.55 - final conf, 0.45 indiv")
display(results6["summary_df"])
print("0.55 - final conf, opt. indiv")
display(results4["summary_df"])

In [ ]:
# --- helper: get the exact IoU key from metrics, tolerant to float quirks ---
def _get_iou_key(metrics: Dict[Any, Any], target: float) -> Optional[float]:
    keys = [k for k in metrics.keys() if isinstance(k, float)]
    if not keys:
        return None
    # prefer exact; else nearest
    for k in keys:
        if abs(k - target) < 1e-9:
            return k
    return min(keys, key=lambda k: abs(k - target))

# --- helper: normalize a grid spec into a list (supports scalar, list, or (start,end,steps)) ---
def _make_grid(spec: Union[float, int, List[Any], tuple]) -> List[Any]:
    # allow explicit None in lists, e.g. for final_nms_iou
    if isinstance(spec, (list, tuple, np.ndarray)):
        if len(spec) == 3 and all(isinstance(x, (int, float)) for x in spec):
            start, end, steps = spec
            if steps <= 1:
                return [float(end)]
            return [float(x) for x in np.linspace(float(start), float(end), int(steps))]
        return [float(x) if x is not None else None for x in spec]
    # scalar
    return [float(spec)]

def sweep_one_parameter(
    *,
    mode: str,                             # 'score' | 'eiou' | 'fnms'
    grid: Union[float, int, List[Any], tuple],
    data_yaml: str,
    models_cfg: List[Dict[str, Any]],
    method: str = "wbf",
    fixed_score_thr: float = 0.0,
    fixed_ensemble_iou: float = 0.55,
    fixed_final_nms_iou: Optional[float] = 0.6,
    # evaluation IoUs
    iou_eval: float = 0.5,                 # which IoU to rank/sort by
    iou_match_list: Optional[List[float]] = None,  # if None, will use [iou_eval]
    data_split: str = "test",
    device: Any = None,
    half: bool = False,
    save_csv_path: Optional[str] = None
) -> pd.DataFrame:
    """
    Sweep exactly one parameter (chosen by 'mode') over 'grid'.
      - mode='score' sweeps score_thr
      - mode='eiou'  sweeps ensemble_iou
      - mode='fnms'  sweeps final_nms_iou (grid may contain None)
    Results table includes TP/FP/FN/P/R/F1/AP at the chosen iou_eval and optional mAP.
    """
    assert mode in {"score", "eiou", "fnms"}, "mode must be 'score', 'eiou', or 'fnms'"
    grid_vals = _make_grid(grid)
    # ensure the evaluation IoU is computed
    iou_list = list(iou_match_list) if iou_match_list is not None else [float(iou_eval)]
    if all(abs(x - iou_eval) > 1e-9 for x in iou_list):
        iou_list = [float(iou_eval)] + iou_list

    records = []
    for val in grid_vals:
        # build args with fixed values
        kwargs = dict(
            data_yaml=data_yaml,
            models_cfg=models_cfg,
            method=method,
            ensemble_iou=float(fixed_ensemble_iou),
            final_nms_iou=fixed_final_nms_iou,
            score_thr=float(fixed_score_thr),
            iou_match_list=iou_list,
            device=device,
            half=half,
            data_split=data_split,
            output_dir=None,   # don't spam disk during sweeps
        )
        # override the one we are sweeping
        if mode == "score":
            kwargs["score_thr"] = float(val)
        elif mode == "eiou":
            kwargs["ensemble_iou"] = float(val)
        elif mode == "fnms":
            kwargs["final_nms_iou"] = None if val is None else float(val)

        res = run_ensemble_evaluation(**kwargs)
        em = res["ensemble_metrics"]
        k = _get_iou_key(em, float(iou_eval))
        if k is None:
            raise RuntimeError(
                f"No IoU={iou_eval:.2f} metrics were computed; "
                f"set iou_match_list to include {iou_eval:.2f}."
            )
        row = {
            mode: val,
            f"TP@{iou_eval:.2f}": int(em[k]["TP"]),
            f"FP@{iou_eval:.2f}": int(em[k]["FP"]),
            f"FN@{iou_eval:.2f}": int(em[k]["FN"]),
            f"P@{iou_eval:.2f}":  float(em[k]["precision"]),
            f"R@{iou_eval:.2f}":  float(em[k]["recall"]),
            f"F1@{iou_eval:.2f}": float(em[k]["f1"]),
            f"AP@{iou_eval:.2f}": float(em[k]["AP"]),
        }
        if "mAP" in em:
            row["mAP"] = float(em["mAP"])
        records.append(row)

    if not records:
        raise RuntimeError("Sweep produced no rows — check your grid and inputs.")

    df = pd.DataFrame.from_records(records).sort_values(f"F1@{iou_eval:.2f}", ascending=False).reset_index(drop=True)
    if save_csv_path:
        df.to_csv(save_csv_path, index=False)
    return df

In [ ]:
grid = [0.49, 0.50, 0.51, 0.52, 0.65]
df_eiou = sweep_one_parameter(
    mode="score",
    grid=grid,
    data_yaml=cfg["sleeve_yaml_path"],
    models_cfg=models_cfg,
    fixed_score_thr=0.55,        # keep score threshold constant
    fixed_final_nms_iou=None,    # no second NMS
    iou_eval=0.10,               # sort/rank at IoU=0.10
    iou_match_list=[0.10],       # compute metrics at 0.10
    data_split="test",
    device=0
)
display(df_eiou)
print("Best ensemble_iou:\n", df_eiou.iloc[0])

In [ ]:
def _linspace(start: float, end: float, steps: int):
    """Inclusive linspace with sane degenerate handling."""
    if steps <= 1:
        return [float(end)]
    return [float(x) for x in np.linspace(float(start), float(end), int(steps))]

def _round2(x):
    return None if x is None else float(np.round(x, 2))

def _mk_iou_threshold_list(start: float, end: float, steps: int):
    """Generate the iou_match_list used to compute AP/mAP."""
    vals = _linspace(start, end, steps)
    # Clamp to [0,1] and round for clean column names
    vals = [float(np.clip(v, 0.0, 1.0)) for v in vals]
    return [float(np.round(v, 2)) for v in vals]

def _safe(o):
    from pathlib import Path
    import numpy as np, os
    if isinstance(o, (np.floating,)):
        return float(o)
    if isinstance(o, (np.integer,)):
        return int(o)
    if isinstance(o, (Path,)):
        return str(o)
    if hasattr(o, "__fspath__"):
        return os.fspath(o)
    return o

In [ ]:
def sweep_ensemble_configs(
    enable_sweep: bool,
    *,
    data_yaml: str,
    models_cfg: list,
    base_output_dir: str = "ensembles/sleeves_v6/config_iteration",
    data_split: str = "test",
    device=None,
    half=False,
    # Search spaces (inclusive) — adjust start/end/steps as you like
    ensemble_iou_range = (0.50, 0.70, 5),
    final_nms_iou_range = (0.55, 0.75, 5),
    include_none_for_final_nms: bool = True,
    score_thr_range = (0.00, 0.20, 10),
    # IoU thresholds used for evaluation (AP/mAP). Not a grid dim; this builds the list.
    iou_match_list_range = (0.50, 0.95, 10),
    # Objective to rank the best row. Options: "F1@0.50", "AP@0.50", "mAP", "F1@0.90", etc.
    objective: str = "F1@0.5",
    method: str = "wbf",
    final_tag_note: str = "",   # optional note added to summary filename
):
    """
    Runs a grid over (ensemble_iou, final_nms_iou, score_thr) and evaluates with
    an IoU threshold list built from iou_match_list_range.
    """
    if not enable_sweep:
        print("[sweep] Disabled (enable_sweep=False). Nothing to do.")
        return None

    base_dir = Path(base_output_dir)
    base_dir.mkdir(parents=True, exist_ok=True)

    # Build grids
    eiou_vals   = [_round2(v) for v in _linspace(*ensemble_iou_range)]
    fnms_vals   = [_round2(v) for v in _linspace(*final_nms_iou_range)]
    if include_none_for_final_nms:
        fnms_vals = [None] + fnms_vals
    score_vals  = [_round2(v) for v in _linspace(*score_thr_range)]
    iou_list    = _mk_iou_threshold_list(*iou_match_list_range)

    print(f"[sweep] Grid sizes: |ensemble_iou|={len(eiou_vals)}  |final_nms_iou|={len(fnms_vals)}  |score_thr|={len(score_vals)}")
    print(f"[sweep] Eval IoUs: {iou_list}")

    rows = []
    iter_idx = 0
    for eiou, fnms, sthr in itertools.product(eiou_vals, fnms_vals, score_vals):
        tag = f"eiou{eiou:.2f}_fnms{('None' if fnms is None else f'{fnms:.2f}')}_score{sthr:.2f}"
        out_dir = base_dir / f"iter_{iter_idx:04d}_{tag}"
        iter_idx += 1

        # Run one configuration
        res = run_ensemble_evaluation(
            data_yaml=data_yaml,
            models_cfg=models_cfg,
            method=method,
            ensemble_iou=eiou,
            final_nms_iou=fnms,
            score_thr=sthr,
            iou_match_list=iou_list,   # AP at these IoUs; mAP is their mean
            device=device,
            half=half,
            data_split=data_split,
            output_dir=str(out_dir),
        )

        # --- extract key stats for IoU=0.50 and 0.90 (if present) ---
        def _get(metrics, thr):
            if thr in metrics:
                m = metrics[thr]
                return {
                    f"TP@{thr:.2f}": m["TP"],
                    f"FP@{thr:.2f}": m["FP"],
                    f"FN@{thr:.2f}": m["FN"],
                    f"P@{thr:.2f}":  m["precision"],
                    f"R@{thr:.2f}":  m["recall"],
                    f"F1@{thr:.2f}": m["f1"],
                    f"AP@{thr:.2f}": m["AP"],
                }
            else:
                return {
                    f"TP@{thr:.2f}": "",
                    f"FP@{thr:.2f}": "",
                    f"FN@{thr:.2f}": "",
                    f"P@{thr:.2f}":  "",
                    f"R@{thr:.2f}":  "",
                    f"F1@{thr:.2f}": "",
                    f"AP@{thr:.2f}": "",
                }

        em = res["ensemble_metrics"]
        row = {
            "iter": iter_idx-1,
            "ensemble_iou": eiou,
            "final_nms_iou": ("" if fnms is None else fnms),
            "score_thr": sthr,
            "split": res.get("split_used", data_split),
            "out_dir": str(out_dir),
        }
        row.update(_get(em, 0.50))
        row.update(_get(em, 0.90))
        row["mAP"] = em.get("mAP", "")
        rows.append(row)

    summary = pd.DataFrame(rows)

    # Rank & save
    if objective not in summary.columns:
        print(f"[sweep] Objective '{objective}' not found in columns. Available: {list(summary.columns)}")
    else:
        best_idx = summary[objective].astype(float).idxmax()
        best_row = summary.loc[best_idx].to_dict()
        with open(base_dir / "best_config.json", "w") as f:
            json.dump(best_row, f, default=_safe, indent=2)
        print(f"[sweep] Best by {objective}: iter={int(best_row['iter'])}  ->  {best_row}")

    # Save the compact sweep table
    suffix = (f"_{final_tag_note}" if final_tag_note else "")
    csv_path = base_dir / f"sweep_summary{suffix}.csv"
    summary.to_csv(csv_path, index=False)
    print(f"[sweep] Wrote {csv_path}")

    return summary

In [ ]:
# Your fixed dataset yaml path and models
data_yaml = dataset_yaml_path

summary_df = sweep_ensemble_configs(
    enable_sweep=True,
    data_yaml=data_yaml,
    models_cfg=models_cfg,
    base_output_dir="ensembles/sleeves_v6/config_iteration",
    data_split="test",
    device=0.0,
    half=False,
    # Narrow ranges first; widen later if needed
    ensemble_iou_range=(0.1, 0.5, 1),      #0.50, 0.55, 0.60, 0.65
    final_nms_iou_range=(0.1, 0.5, 1),     #(plus None if include_none_for_final_nms=True)
    include_none_for_final_nms=False,
    score_thr_range=(0.42, 0.62, 10),        #0.00, 0.05, 0.10, 0.15, 0.20
    iou_match_list_range=(0.1, 0.90, 3),   #AP@0.50..0.95 for mAP
    objective="F1@0.10",                     #or "AP@0.50", "mAP", "F1@0.90"
    method="wbf",
    final_tag_note="trialB"
)
display(summary_df.sort_values("F1@0.50", ascending=False).head(10))